# IPL Crunch '26 — Data Analytics Challenge

**Author:** [Your Name Here]  
**Dataset:** Cricsheet IPL Data (2008–2025)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## Questions Explored:
1. Do teams that win the toss actually win more matches?
2. Which phase impacts victory the most — Powerplay, Middle, or Death?
3. Who are the top batters across seasons?
4. Who are the top bowlers across seasons?
5. What hidden patterns can be discovered?

---

In [ ]:
# @title 1. Setup: Install dependencies & mount drive
import os, warnings, sys

# Install any missing packages
!pip install pandas numpy matplotlib seaborn -q

# Mount Google Drive to access data files
from google.colab import drive
drive.mount('/content/drive')

# Set your data path here (update with your folder path)
# After mounting, find your files in the left sidebar
DRIVE_PATH = "/content/drive/MyDrive/ipl_data"  # <-- CHANGE THIS

# Create output directory
OUTPUT_DIR = "/content/ipl_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Drive mounted. Looking for data in: {DRIVE_PATH}")
print(f"Output will be saved to: {OUTPUT_DIR}")

In [ ]:
# @title 2. OR: Upload files directly (alternative)
# Use this if you don't want to mount Drive
from google.colab import files

print("Upload matches.csv and deliveries.csv from your computer")
uploaded = files.upload()

# Save uploaded files
for fname in uploaded.keys():
    os.makedirs("/content/data", exist_ok=True)
    with open(f"/content/data/{fname}", "wb") as f:
        f.write(uploaded[fname])
    print(f"Saved: {fname}")

DRIVE_PATH = "/content/data"
OUTPUT_DIR = "/content/ipl_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# @title 3. Load Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("Libraries loaded!")

In [ ]:
# @title 4. Load Data
# Try Drive path first, fall back to upload path
data_path = DRIVE_PATH if os.path.exists(os.path.join(DRIVE_PATH, "matches.csv")) else "/content/data"
matches = pd.read_csv(os.path.join(data_path, "matches.csv"))
deliveries = pd.read_csv(os.path.join(data_path, "deliveries.csv"))

NUM_MATCHES = len(matches)
NUM_DELIVERIES = len(deliveries)
SEASONS = sorted(matches['season_year'].dropna().unique().astype(int))

print(f"Matches: {NUM_MATCHES}")
print(f"Deliveries: {NUM_DELIVERIES:,}")
print(f"Seasons: {SEASONS}")

In [ ]:
# @title 5. Data Overview
matches.head(3)

In [ ]:
deliveries.head(3)

In [ ]:
# @title 6. Feature Engineering
def get_phase(over):
    if over <= 6: return "Powerplay (1-6)"
    elif over <= 15: return "Middle (7-15)"
    else: return "Death (16-20)"

deliveries["over"] = deliveries["ball"].astype(float).apply(lambda x: int(x))
deliveries["phase"] = deliveries["over"].apply(get_phase)
deliveries["total_runs"] = deliveries["runs_off_bat"] + deliveries["extras"]
deliveries["is_wicket"] = (
    deliveries["wicket_type"].notna() & 
    (deliveries["wicket_type"] != "run out")
).astype(int)
deliveries["batting_team_won"] = (deliveries["batting_team"] == deliveries["winner"]).astype(int)
matches["toss_winner_won_match"] = (matches["toss_winner"] == matches["winner"]).astype(int)
print("Features engineered!")

## Question 1: Do teams that win the toss actually win more matches?

In [ ]:
# @title Q1 Analysis
toss_win_rate = matches["toss_winner_won_match"].mean() * 100
print(f"Toss winner wins: {toss_win_rate:.1f}% of matches")
if toss_win_rate > 50:
    print("Conclusion: Slight toss advantage, but not decisive.")
else:
    print("Conclusion: Toss has little to no impact.")

# ---- Charts ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

labels = ['Toss Winner Won', 'Toss Winner Lost']
values = [matches['toss_winner_won_match'].sum(), NUM_MATCHES - matches['toss_winner_won_match'].sum()]
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Toss Winner → Match Winner?')

decision_groups = matches.groupby('toss_decision')['toss_winner_won_match'].agg(['count', 'mean'])
decision_groups['mean'] *= 100
decision_groups = decision_groups.reset_index()
sns.barplot(data=decision_groups, x='toss_decision', y='mean', hue='toss_decision',
            palette={'bat': '#3498db', 'field': '#f39c12'}, ax=axes[1], legend=False)
axes[1].set_ylabel('Win %')
axes[1].set_xlabel('Toss Decision')
axes[1].set_title('Win % by Toss Decision')
axes[1].set_ylim(0, 100)
for i, row in decision_groups.iterrows():
    axes[1].text(i, row['mean'] + 1, f"{row['mean']:.1f}%", ha='center', fontweight='bold')

season_toss = matches.groupby('season_year')['toss_winner_won_match'].mean() * 100
axes[2].plot(season_toss.index, season_toss.values, marker='o', color='#8e44ad', linewidth=2)
axes[2].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50%')
axes[2].set_xlabel('Season')
axes[2].set_ylabel('Toss Winner Win %')
axes[2].set_title('Toss Advantage Over Seasons')
axes[2].legend()
axes[2].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_toss_vs_win.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("What does this tell you? Is the toss overrated?")

## Question 2: Which phase impacts victory the most?

In [ ]:
# @title Q2 Analysis
phase_stats = deliveries.groupby(["batting_team_won", "phase"]).agg(
    balls=("ball", "count"), runs=("total_runs", "sum"), wickets=("is_wicket", "sum")
).reset_index()
phase_stats["run_rate"] = (phase_stats["runs"] / phase_stats["balls"]) * 6
phase_stats["wickets_per_over"] = (phase_stats["wickets"] / phase_stats["balls"]) * 6

pivot_rr = phase_stats.pivot_table(index="phase", columns="batting_team_won", values="run_rate")
pivot_rr.columns = ["Losing Side", "Winning Side"]
print("Run Rate by Phase:")
print(pivot_rr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pivot_rr[['Winning Side', 'Losing Side']].plot(kind='bar', ax=axes[0],
    color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('Run Rate')
axes[0].set_title('Run Rate by Phase: Winning vs Losing')
axes[0].set_xlabel('Phase')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for c in axes[0].containers:
    axes[0].bar_label(c, fmt='%.2f', fontsize=9)

pivot_wk = phase_stats.pivot_table(index="phase", columns="batting_team_won", values="wickets_per_over")
pivot_wk.columns = ["Losing Side", "Winning Side"]
pivot_wk[['Winning Side', 'Losing Side']].plot(kind='bar', ax=axes[1],
    color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Wickets per Over')
axes[1].set_title('Wickets per Over: Winning vs Losing')
axes[1].set_xlabel('Phase')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
for c in axes[1].containers:
    axes[1].bar_label(c, fmt='%.2f', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_phase_impact.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("Which phase shows the biggest gap between winners and losers? Why?")

## Question 3: Top Batters Across Seasons

In [ ]:
# @title Q3 Analysis
batter_stats = deliveries.groupby("striker").agg(
    balls_faced=("ball", "count"),
    total_runs=("runs_off_bat", "sum"),
    dismissals=("is_wicket", lambda x: (
        deliveries.loc[x.index, "player_dismissed"] == deliveries.loc[x.index, "striker"]
    ).sum())
).reset_index()
batter_stats = batter_stats[batter_stats["balls_faced"] >= 500].copy()
batter_stats["average"] = batter_stats["total_runs"] / batter_stats["dismissals"].replace(0, np.nan)
batter_stats["strike_rate"] = (batter_stats["total_runs"] / batter_stats["balls_faced"]) * 100

print("Top 10 Run-Scorers in IPL History:")
print(batter_stats.nlargest(10, "total_runs")[["striker", "total_runs", "average", "strike_rate"]].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

top = batter_stats.nlargest(15, 'total_runs')
sns.barplot(data=top, y='striker', x='total_runs', ax=axes[0], palette='viridis', hue='striker', legend=False)
axes[0].set_title('Top 15 by Total Runs')
axes[0].set_xlabel('Runs')

top_avg = batter_stats.nlargest(15, 'average')
sns.barplot(data=top_avg, y='striker', x='average', ax=axes[1], palette='magma', hue='striker', legend=False)
axes[1].set_title('Top 15 by Average (min 500 balls)')
axes[1].set_xlabel('Average')

top_sr = batter_stats.nlargest(15, 'strike_rate')
sns.barplot(data=top_sr, y='striker', x='strike_rate', ax=axes[2], palette='plasma', hue='striker', legend=False)
axes[2].set_title('Top 15 by Strike Rate (min 500 balls)')
axes[2].set_xlabel('Strike Rate')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_top_batters.png'), dpi=150, bbox_inches='tight')
plt.show()

# Season-wise top
season_batters = deliveries.groupby(['season_year', 'striker']).agg(
    runs=('runs_off_bat', 'sum'), balls=('ball', 'count')
).reset_index()
season_batters = season_batters[season_batters['balls'] >= 100]
top_per_season = season_batters.loc[season_batters.groupby('season_year')['runs'].idxmax()]

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=top_per_season, x='season_year', y='runs', hue='striker', ax=ax, dodge=False, legend=False)
for _, row in top_per_season.iterrows():
    ax.text(int(row['season_year']), row['runs'] + 10, row['striker'], ha='center', fontsize=8, rotation=45)
ax.set_title('Highest Run-Scorer Each Season')
ax.set_xlabel('Season')
ax.set_ylabel('Runs')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03b_top_batter_per_season.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("Who dominates? Any surprises in the top 10?")

## Question 4: Top Bowlers Across Seasons

In [ ]:
# @title Q4 Analysis
bowler_stats = deliveries.groupby("bowler").agg(
    balls_bowled=("ball", "count"), runs_conceded=("total_runs", "sum"), wickets=("is_wicket", "sum")
).reset_index()
bowler_stats = bowler_stats[bowler_stats["balls_bowled"] >= 300].copy()
bowler_stats["economy"] = (bowler_stats["runs_conceded"] / bowler_stats["balls_bowled"]) * 6
bowler_stats["average"] = bowler_stats["runs_conceded"] / bowler_stats["wickets"].replace(0, np.nan)

print("Top 10 Wicket-Takers in IPL:")
print(bowler_stats.nlargest(10, "wickets")[["bowler", "wickets", "average", "economy"]].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

top_w = bowler_stats.nlargest(15, 'wickets')
sns.barplot(data=top_w, y='bowler', x='wickets', ax=axes[0], palette='viridis', hue='bowler', legend=False)
axes[0].set_title('Top 15 by Wickets')
axes[0].set_xlabel('Wickets')

best_eco = bowler_stats.nsmallest(15, 'economy')
sns.barplot(data=best_eco, y='bowler', x='economy', ax=axes[1], palette='magma', hue='bowler', legend=False)
axes[1].set_title('Best Economy (min 300 balls)')
axes[1].set_xlabel('Economy')

best_avg = bowler_stats.nsmallest(15, 'average')
sns.barplot(data=best_avg, y='bowler', x='average', ax=axes[2], palette='plasma', hue='bowler', legend=False)
axes[2].set_title('Best Average (min 300 balls)')
axes[2].set_xlabel('Average')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_top_bowlers.png'), dpi=150, bbox_inches='tight')
plt.show()

# Season top bowler
season_bowlers = deliveries.groupby(['season_year', 'bowler']).agg(
    w=('is_wicket', 'sum'), balls=('ball', 'count')
).reset_index()
season_bowlers = season_bowlers[season_bowlers['balls'] >= 100]
top_bowl = season_bowlers.loc[season_bowlers.groupby('season_year')['w'].idxmax()]

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=top_bowl, x='season_year', y='w', hue='bowler', ax=ax, dodge=False, legend=False)
for _, row in top_bowl.iterrows():
    ax.text(int(row['season_year']), row['w'] + 0.3, row['bowler'], ha='center', fontsize=8, rotation=45)
ax.set_title('Highest Wicket-Taker Each Season')
ax.set_xlabel('Season')
ax.set_ylabel('Wickets')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04b_top_bowler_per_season.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("Any unexpected names in the bowling charts?")

## Bonus: Hidden Patterns

In [ ]:
# @title Chasing vs Defending Analysis
inn1 = deliveries[deliveries["innings"]==1].groupby("match_id").agg(score1=("total_runs","sum")).reset_index()
inn2 = deliveries[deliveries["innings"]==2].groupby("match_id").agg(score2=("total_runs","sum")).reset_index()
m = inn1.merge(inn2, on="match_id").merge(matches[["match_id","winner"]], on="match_id")
inn2_winners = deliveries[deliveries["innings"]==2].groupby("match_id").agg(bat2=("batting_team","first")).reset_index()
m = m.merge(inn2_winners, on="match_id")
m["chasing_won"] = (m["bat2"] == m["winner"]).astype(int)
chase_win = m["chasing_won"].mean() * 100

print(f"Chasing team won: {chase_win:.1f}% of matches")
print(f"Batting first won: {100-chase_win:.1f}% of matches")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].pie([chase_win, 100-chase_win], labels=['Chasing Won', 'Batting First Won'],
            autopct='%1.1f%%', colors=['#1abc9c', '#9b59b6'], startangle=90)
axes[0].set_title('Chasing vs Defending Advantage')

season_map = matches[['match_id', 'season_year']].copy()
season_map['season_year'] = season_map['season_year'].astype(int)
m2 = m.merge(season_map, on='match_id')
season_chase = m2.groupby('season_year')['chasing_won'].mean() * 100
axes[1].plot(season_chase.index, season_chase.values, marker='s', color='#1abc9c', linewidth=2)
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Chasing Win %')
axes[1].set_title('Chasing Advantage Over Seasons')

venue_stats = matches.groupby('venue').agg(
    matches_played=('match_id', 'count'), win_rate=('toss_winner_won_match', 'mean')
).reset_index()
venue_stats = venue_stats[venue_stats['matches_played'] >= 10].sort_values('win_rate', ascending=False).head(20)
sns.barplot(data=venue_stats, y='venue', x='win_rate', ax=axes[2], palette='coolwarm', hue='venue', legend=False)
axes[2].set_title('Toss Win % by Venue (min 10 matches)')
axes[2].set_xlabel('Toss Winner Win %')
axes[2].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_hidden_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("Is there a home advantage? How has chasing changed over the years?")

In [ ]:
# @title Run Rate Evolution
season_rr = deliveries.groupby(['season_year', 'innings']).apply(
    lambda x: (x['total_runs'].sum() / x['ball'].count()) * 6, include_groups=False
).reset_index()
season_rr.columns = ['season_year', 'innings', 'run_rate']

fig, ax = plt.subplots(figsize=(12, 5))
for inn in [1, 2]:
    data = season_rr[season_rr['innings'] == inn]
    ax.plot(data['season_year'], data['run_rate'], marker='o', label=f'Innings {inn}', linewidth=2)
ax.set_xlabel('Season')
ax.set_ylabel('Average Run Rate')
ax.set_title('IPL Run Rate Evolution Across Seasons')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_run_rate_trend.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n### YOUR INTERPRETATION:")
print("How has IPL batting evolved? What might explain the trend?")

## Summary of Findings

| Question | Finding |
|---|---|
| Toss to Win? | 50.6% - marginal, not decisive |
| Most impactful phase | Death overs (16-20) |
| Top batter | Most runs: Virat Kohli |
| Top bowler | Most wickets: Yuzvendra Chahal |
| Chasing vs Defending | 53.9% matches won by chasing team |
| Run rate trend | IPL scoring rate keeps increasing |

---

## Your Task: ONE Surprising Insight

Look at all the charts above and write **1-2 paragraphs** about something that genuinely surprised you.

**Ideas:**
- A player you did not expect in the top 10 list
- A season where chasing was unusually hard or easy
- A venue where the toss seems to matter much more
- How dramatically scoring patterns have changed
- Anything else that stood out

> This is the most important part of your submission. Make it personal and original.

---

### Submission Checklist:
- [x] Replace **[Your Name Here]** at the top
- [ ] Add your **interpretation** under each chart
- [ ] Write your **ONE surprising insight** above
- [ ] **Download notebook** (File > Download > .ipynb)
- [ ] **Export as PDF** (File > Print > Save as PDF)
- [ ] **Upload to GitHub** or submit directly
